### Global-level

Global Pearson

In [1]:
import json
import itertools
from pathlib import Path
import pandas as pd
import os
import numpy as np
from scipy.stats import pearsonr  # Essential for the new correlation logic

# --- Configuration ---
OUTPUT_ROOT = "/Users/javigg/Desktop/h2t_correlations/evaluation/output_evals"

DATASETS = [
    "fleurs", "covost2", "europarl_st", "cs_fleurs", "mexpresso", 
    "acl6060-short", "acl6060-long", "mcif-long", "mcif-short"
]

DIRECTION_PAIRS = [
    "en_de","de_en","en_es","es_en","en_fr","fr_en","en_it","it_en",
    "en_nl","nl_en","en_pt","pt_en","en_zh","zh_en"
]

SYSTEM_NAMES = ['whisper', 'seamlessm4t', 'canary-v2', 'owsm4.0-ctc',
                'aya_whisper', 'gemma_whisper', 'tower_whisper', 
                'aya_seamlessm4t', 'gemma_seamlessm4t', 'tower_seamlessm4t',
                'aya_canary-v2', 'gemma_canary-v2', 'tower_canary-v2',
                'aya_owsm4.0-ctc', 'gemma_owsm4.0-ctc', 'tower_owsm4.0-ctc',
                'desta2-8b', 'qwen2audio-7b', 'phi4multimodal', 'voxtral-small-24b', 'spirelm'
               ]

SELECTED_COLS = [
    "dataset", "direction", "system",
    "SacreBLEU", "chrF", "LinguaPy",
    "RefMetricX_24", "RefMetricX_24-Strict-linguapy", "QEMetricX_24-Strict-linguapy",
    "XCOMET-Strict-linguapy", "XCOMET", "XCOMET-QE-Strict-linguapy", "QEMetricX_24", "XCOMET-QE"
]

METRICX_REF = "RefMetricX_24-Strict-linguapy"
METRICX_QE  = "QEMetricX_24-Strict-linguapy"
XCOMET_REF  = "XCOMET-Strict-linguapy"
XCOMET_QE   = "XCOMET-QE-Strict-linguapy"

# --- Data Loading Functions ---

def load_results_summaries(base_dir, direction_pairs, system_names):
    base_path = Path(base_dir)
    all_results = {}

    for direction, system in itertools.product(direction_pairs, system_names):
        summary_path = base_path / system / direction / 'results_summary.jsonl'
        
        if direction not in all_results:
            all_results[direction] = {}

        try:
            with summary_path.open('r', encoding='utf-8') as f:
                all_results[direction][system] = [json.loads(line) for line in f]
        except FileNotFoundError:
            all_results[direction][system] = None
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON in {summary_path}: {e}")
            all_results[direction][system] = None

    return all_results

def convert_results_to_dataframe(results_data):
    all_records = [
        {
            'direction': direction,
            'system': system,
            **record 
        }
        for direction, systems in results_data.items()
        for system, records in systems.items()
        if records is not None
        for record in records
    ]

    if not all_records:
        return pd.DataFrame()

    df = pd.DataFrame(all_records)
    original_cols = [col for col in df.columns if col not in ['direction', 'system']]
    preferred_order = ['direction', 'system'] + original_cols
    return df[preferred_order]

def load_dataset_df(dataset_name: str) -> pd.DataFrame:
    base_dir = os.path.join(OUTPUT_ROOT, dataset_name)
    results_data = load_results_summaries(base_dir, DIRECTION_PAIRS, SYSTEM_NAMES)
    results_df = convert_results_to_dataframe(results_data)
    results_df = results_df.copy()
    results_df["dataset"] = dataset_name
    return results_df

# --- Correlation Logic ---

def compute_qe_ref_correlations(df: pd.DataFrame,
                                dataset_col: str = "dataset",
                                direction_col: str = "direction") -> pd.DataFrame:
    """
    Computes Pearson correlation and p-values using scipy.stats.pearsonr.
    """
    results = []
    group_cols = [dataset_col, direction_col] if dataset_col in df.columns else [direction_col]

    for keys, sub in df.groupby(group_cols):
        if isinstance(keys, tuple):
            dataset, direction = keys
        else:
            dataset, direction = (None, keys)

        def get_stats(data, col_ref, col_qe):
            # Drop rows where either metric is missing
            valid_sub = data[[col_ref, col_qe]].dropna()
            # Pearson requires at least 2 data points for a correlation
            if len(valid_sub) > 1:
                # scipy returns (correlation_coefficient, p_value)
                r_val, p_val = pearsonr(valid_sub[col_ref], valid_sub[col_qe])
                return r_val, p_val, len(valid_sub)
            return np.nan, np.nan, len(valid_sub)

        mx_r, mx_p, mx_n = get_stats(sub, METRICX_REF, METRICX_QE)
        xc_r, xc_p, xc_n = get_stats(sub, XCOMET_REF, XCOMET_QE)

        results.append({
            "dataset": dataset if dataset_col in df.columns else "ALL",
            "language_pair": direction,
            "mx_pearson": mx_r,
            "mx_p_val": mx_p,
            "xc_pearson": xc_r,
            "xc_p_val": xc_p,
            "n_mx": mx_n,
            "n_xc": xc_n,
        })

    return pd.DataFrame(results).sort_values(["dataset", "language_pair"])

# --- Main Execution ---

all_dfs = []
for ds in DATASETS:
    try:
        all_dfs.append(load_dataset_df(ds))
    except Exception as e:
        print(f"[WARN] Failed loading dataset={ds}: {e}")

if not all_dfs:
    raise RuntimeError("No datasets were loaded.")

results_df_all = pd.concat(all_dfs, ignore_index=True)
existing_cols = [c for c in SELECTED_COLS if c in results_df_all.columns]
df_filtered = results_df_all[existing_cols].copy()

# Compute correlations with scipy
corr_table = compute_qe_ref_correlations(df_filtered)

print("### Detailed Pearson Correlations (per Dataset & Direction) ###")
with pd.option_context("display.max_rows", None, "display.precision", 4):
    print(corr_table.to_string(index=False))

# Dataset-level summary with Significance Count
summary = (
    corr_table
    .groupby("dataset", as_index=False)
    .agg(
        avg_mx_pearson=("mx_pearson", "mean"),
        mx_sig_count=("mx_p_val", lambda x: (x < 0.05).sum()),
        avg_xc_pearson=("xc_pearson", "mean"),
        xc_sig_count=("xc_p_val", lambda x: (x < 0.05).sum()),
        total_directions=("language_pair", "count")
    )
    .sort_values("dataset")
)

print("\n=== Dataset-level summary (Mean Correlation & Significance) ===")
print(summary.to_string(index=False))

### Detailed Pearson Correlations (per Dataset & Direction) ###
      dataset language_pair  mx_pearson   mx_p_val  xc_pearson   xc_p_val  n_mx  n_xc
 acl6060-long         en_de      0.9996 1.3139e-21      0.9996 9.0882e-22    15    15
 acl6060-long         en_fr      0.9997 1.5636e-22      0.9996 1.3106e-21    15    15
 acl6060-long         en_pt      0.9999 3.8433e-26      0.9998 3.4121e-24    15    15
 acl6060-long         en_zh      0.9999 4.4415e-25      1.0000 1.6066e-27    15    15
acl6060-short         en_de      0.9989 3.0652e-25      0.9986 1.4902e-24    20    20
acl6060-short         en_fr      0.9993 5.2224e-27      0.9987 1.2034e-24    20    20
acl6060-short         en_pt      0.9991 5.6772e-26      0.9980 5.6801e-23    20    20
acl6060-short         en_zh      0.9999 2.6894e-35      0.9998 9.9139e-33    20    20
      covost2         de_en      0.9989 2.7986e-25      0.9970 2.1000e-21    20    20
      covost2         en_de      0.9995 7.5081e-29      0.9991 2.6887e-26   

### Item level

Group-by-item Spearman

In [2]:
import json
import itertools
from pathlib import Path
import pandas as pd
import os
import numpy as np
from scipy.stats import spearmanr, pearsonr

# --- Configuration ---
OUTPUT_ROOT = "/Users/javigg/Desktop/h2t_correlations/evaluation/output_evals"
MANIFEST_ROOT = "/Users/javigg/Desktop/h2t_correlations/manifests"

DATASETS = [
    "fleurs", "covost2", "europarl_st", "cs_fleurs", "mexpresso",
    "acl6060-short", "acl6060-long", "mcif-long", "mcif-short"
]

DIRECTION_PAIRS = [
    "en_de","de_en","en_es","es_en","en_fr","fr_en","en_it","it_en",
    "en_nl","nl_en","en_pt","pt_en","en_zh","zh_en"
]

SYSTEM_NAMES = [
    'whisper', 'seamlessm4t', 'canary-v2', 'owsm4.0-ctc',
    'aya_whisper', 'gemma_whisper', 'tower_whisper',
    'aya_seamlessm4t', 'gemma_seamlessm4t', 'tower_seamlessm4t',
    'aya_canary-v2', 'gemma_canary-v2', 'tower_canary-v2',
    'aya_owsm4.0-ctc', 'gemma_owsm4.0-ctc', 'tower_owsm4.0-ctc',
    'desta2-8b', 'qwen2audio-7b', 'phi4multimodal', 'voxtral-small-24b', 'spirelm'
]

METRICX_REF = "metricx_score"
METRICX_QE  = "metricx_qe_score"
XCOMET_REF  = "xcomet_score"
XCOMET_QE   = "xcomet_qe_score"

# --- Data Loading Functions ---

def load_all_jsons(base_dir, manifests_dir, direction_pairs, system_names):
    """
    Loads item-level scores for all systems and directions in a specific dataset.
    NOTE: REF metrics are set to NaN if not present, so we can exclude directions
    where ref-based scores do not exist.
    """
    base_path = Path(base_dir)
    all_results = {}

    for direction, system in itertools.product(direction_pairs, system_names):
        results_path = base_path / system / direction / 'results.jsonl'

        if direction not in all_results:
            all_results[direction] = {}

        try:
            if not results_path.exists():
                continue

            with results_path.open('r', encoding='utf-8') as f:
                system_results = [json.loads(line) for line in f]

            for it in system_results:
                linguapy_val = it['metrics']['linguapy_score']
                it['linguapy_score'] = linguapy_val[0] if isinstance(linguapy_val, list) else linguapy_val

                is_valid_lang = (it['linguapy_score'] == 0)

                # QE metrics: keep your existing fallback behavior
                it[XCOMET_QE]   = it['metrics'].get(XCOMET_QE, 0) if is_valid_lang else 0
                it[METRICX_QE]  = it['metrics'].get(METRICX_QE, 0) if is_valid_lang else 25

                # REF metrics: IMPORTANT CHANGE
                # If ref wasn't computed, it should be NaN (not 0), so we can detect "missing refs".
                it[XCOMET_REF]  = it['metrics'].get(XCOMET_REF, np.nan)

                if not pd.isna(it[XCOMET_REF]) and not is_valid_lang:
                    it[XCOMET_REF] = 0
                
                it[METRICX_REF] = it['metrics'].get(METRICX_REF, np.nan)

                if not pd.isna(it[METRICX_REF]) and not is_valid_lang:
                    it[METRICX_REF] = 25

            all_results[direction][system] = system_results

        except Exception as e:
            print(f"Error processing {system}/{direction}: {e}")
            continue

    # Flatten to list
    results = []
    for direction in all_results:
        for system in all_results[direction]:
            for item in all_results[direction][system]:
                item['direction'] = direction
                item['system'] = system
                results.append(item)

    return pd.DataFrame(results)

# --- Correlation Logic ---

def is_constant(series: pd.Series) -> bool:
    return series.nunique(dropna=True) <= 1

def compute_item_level_correlations(
    df: pd.DataFrame,
    dataset_col: str = "dataset",
    direction_col: str = "direction",
    require_ref_coverage: bool = True,
    min_items_with_ref: int = 1,
) -> pd.DataFrame:
    """
    Group-by-item Spearman correlation between REF vs QE (system ranks per item).
    Excludes directions where REF-based scores do not exist.
    """
    results = []
    group_cols = [dataset_col, direction_col] if dataset_col in df.columns else [direction_col]

    def get_byitem_spearman(data: pd.DataFrame, col_ref: str, col_qe: str) -> float:
        if col_ref not in data.columns or col_qe not in data.columns:
            return np.nan

        # Only keep rows where BOTH ref and qe exist
        data = data[[ "sample_id", "system", col_ref, col_qe ]].copy()
        data = data.dropna(subset=[col_ref, col_qe])

        # If there is no ref coverage (after filtering), caller will handle exclusion
        if data.empty:
            return np.nan

        item_corrs = []
        for sample_id, group in data.groupby("sample_id"):
            valid_group = group[[col_ref, col_qe]].dropna()

            # Need >= 2 systems for that item
            if len(valid_group) < 2:
                continue

            if is_constant(valid_group[col_ref]) or is_constant(valid_group[col_qe]):
                item_corrs.append(0.0)
            else:
                corr, _ = spearmanr(valid_group[col_ref], valid_group[col_qe])
                item_corrs.append(corr)

        return np.mean(item_corrs) if item_corrs else np.nan

    for keys, sub_df in df.groupby(group_cols):
        if isinstance(keys, tuple):
            dataset, direction = keys
        else:
            dataset, direction = (None, keys)

        # --- EXCLUDE directions with missing REF metrics ---
        # Count how many rows actually have a REF value (before item-level filtering)
        mx_ref_count = sub_df[METRICX_REF].notna().sum() if METRICX_REF in sub_df.columns else 0
        xc_ref_count = sub_df[XCOMET_REF].notna().sum() if XCOMET_REF in sub_df.columns else 0

        if require_ref_coverage and (mx_ref_count < min_items_with_ref or xc_ref_count < min_items_with_ref):
            # If you want to be stricter/looser, adjust logic above.
            # This matches your ask: directions where ref-based scores won't exist -> exclude.
            continue

        mx_spearman = get_byitem_spearman(sub_df, METRICX_REF, METRICX_QE)
        xc_spearman = get_byitem_spearman(sub_df, XCOMET_REF, XCOMET_QE)

        # Optional: if after filtering there's no valid pairs, skip
        if require_ref_coverage and (np.isnan(mx_spearman) or np.isnan(xc_spearman)):
            continue

        # For reporting, compute counts on the filtered (ref+qe present) subsets
        mx_valid_rows = sub_df[[METRICX_REF, METRICX_QE]].dropna().shape[0] if all(c in sub_df.columns for c in [METRICX_REF, METRICX_QE]) else 0
        xc_valid_rows = sub_df[[XCOMET_REF, XCOMET_QE]].dropna().shape[0] if all(c in sub_df.columns for c in [XCOMET_REF, XCOMET_QE]) else 0

        results.append({
            "dataset": dataset,
            "language_pair": direction,
            "mx_byitem_spearman": mx_spearman,
            "xc_byitem_spearman": xc_spearman,
            "n_unique_samples": sub_df["sample_id"].nunique(),
            "total_rows": len(sub_df),
            "mx_ref_rows": int(mx_ref_count),
            "xc_ref_rows": int(xc_ref_count),
            "mx_valid_ref_qe_rows": int(mx_valid_rows),
            "xc_valid_ref_qe_rows": int(xc_valid_rows),
        })

    if not results:
        return pd.DataFrame()

    return pd.DataFrame(results).sort_values(["dataset", "language_pair"])

# --- Main Execution ---
all_dfs = []

print(f"Starting data load from {OUTPUT_ROOT}...")

for ds in DATASETS:
    ds_path = os.path.join(OUTPUT_ROOT, ds)
    ds_manifest_path = os.path.join(MANIFEST_ROOT, ds)
    final_manifest_path = ds_manifest_path if os.path.exists(ds_manifest_path) else MANIFEST_ROOT

    try:
        print(f"Loading {ds}...")
        df_ds = load_all_jsons(ds_path, final_manifest_path, DIRECTION_PAIRS, SYSTEM_NAMES)

        if not df_ds.empty:
            df_ds["dataset"] = ds
            all_dfs.append(df_ds)
        else:
            print(f" [WARN] No data found for dataset: {ds}")

    except Exception as e:
        print(f" [ERROR] Failed loading dataset={ds}: {e}")

if not all_dfs:
    raise RuntimeError("No datasets were loaded. Check paths and JSON structure.")

print("Concatenating all data...")
results_df_all = pd.concat(all_dfs, ignore_index=True)

print(f"Computing Group-by-Item Spearman on {len(results_df_all)} rows...")
corr_table = compute_item_level_correlations(results_df_all)

print("\n### Group-by-Item Spearman Correlations (Mean per Dataset/Direction) ###")
with pd.option_context("display.max_rows", None, "display.precision", 4):
    print(corr_table.to_string(index=False))

# Dataset-level summary
summary = (
    corr_table
    .groupby("dataset", as_index=False)
    .agg(
        avg_mx_spearman=("mx_byitem_spearman", "mean"),
        avg_xc_spearman=("xc_byitem_spearman", "mean"),
        total_directions=("language_pair", "count"),
        total_samples=("n_unique_samples", "sum"),
    )
    .sort_values("dataset")
)

print("\n=== Dataset-level summary (Global Average) ===")
print(summary.to_string(index=False))

Starting data load from /Users/javigg/Desktop/h2t_correlations/evaluation/output_evals...
Loading fleurs...
Loading covost2...
Loading europarl_st...
Loading cs_fleurs...
Loading mexpresso...
Loading acl6060-short...
Loading acl6060-long...
Loading mcif-long...
Loading mcif-short...
Concatenating all data...
Computing Group-by-Item Spearman on 4611211 rows...

### Group-by-Item Spearman Correlations (Mean per Dataset/Direction) ###
      dataset language_pair  mx_byitem_spearman  xc_byitem_spearman  n_unique_samples  total_rows  mx_ref_rows  xc_ref_rows  mx_valid_ref_qe_rows  xc_valid_ref_qe_rows
 acl6060-long         en_de              0.9112              0.9122               416        6240         6240         6240                  6240                  6240
 acl6060-long         en_fr              0.9201              0.9066               416        6240         6240         6240                  6240                  6240
 acl6060-long         en_pt              0.9183             